In [ ]:
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip -q install -U transformers datasets accelerate evaluate scikit-learn


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 131.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 130.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 51.2 MB/s eta 0:00:00


In [ ]:
!pip -q install optuna


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 9.1 MB/s eta 0:00:00


In [ ]:
!pip -q uninstall -y pyarrow
!pip -q install -U "pyarrow==14.0.2" "pandas==2.2.2" "numpy==1.26.4" datasets


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.0/38.0 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 129.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.1/542.1 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 20.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.36.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cudf-cu12 25.10.0 requires pyarrow>=15.0.0; platform_machine == "x86_64", but you have pyarrow 14.0.2 which is incompati

In [ ]:
import optuna
from transformers import set_seed
import numpy as np
import itertools
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import f1_score, accuracy_score

# -----------------------
# A) Load JSONL
# -----------------------
data_files = {
    "train": "/content/drive/MyDrive/nlpPro/data3/HebNLI_train.balanced.jsonl",
    "validation": "/content/drive/MyDrive/nlpPro/data3/HebNLI_val.full.clean.jsonl",
    "test": "/content/drive/MyDrive/nlpPro/data3/HebNLI_test.full.clean.jsonl",
}
ds = load_dataset("json", data_files=data_files)

# -----------------------
# B) Label mapping
# -----------------------
label_list = ["entailment", "contradiction", "neutral"]
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

def add_label_id(example):
    example["label"] = label2id[str(example["original_label"]).lower()]
    return example

ds = ds.map(add_label_id)

# -----------------------
# C) Tokenizer
# -----------------------
model_name = "dicta-il/dictabert"  # אם זה לא נמצא אצלכם, תגידו לי מה שם המודל המדויק שאתם רוצים
tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})

def tokenize_pair(example):
    return tokenizer(
        example["translation1"],
        example["translation2"],
        truncation=True,
        max_length=128,   # <-- keep fixed during tuning (you tested 256 already)
    )

ds_tok = ds.map(tokenize_pair, batched=False)
cols = ["input_ids", "attention_mask", "label"]
ds_tok = ds_tok.remove_columns([c for c in ds_tok["train"].column_names if c not in cols])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# -----------------------
# D) Metrics
# -----------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

# -----------------------
# E) Model init (fresh model per run)
# -----------------------
def model_init():
    m = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=3,
        id2label=id2label,
        label2id=label2id,
    )
    # if we added pad token, keep embeddings consistent
    if len(tokenizer) != m.config.vocab_size:
        m.resize_token_embeddings(len(tokenizer))
    return m

# -----------------------
# F) GRID SEARCH
# -----------------------
set_seed(42)

def hp_space(trial: optuna.Trial):
    # You can keep the exact same values you had in grid search (categorical),
    # or make some of them continuous (suggest_float(..., log=True)).
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.05),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [16, 32]),
        "warmup_ratio": trial.suggest_float("warmup_ratio", 0.0, 0.1),
        "num_train_epochs": trial.suggest_categorical("num_train_epochs", [2, 3]),
    }

def compute_objective(metrics):
    # HF will pass eval metrics dict here
    return metrics["eval_macro_f1"]

base_args = TrainingArguments(
    output_dir="dictabert_optuna",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    report_to="none",
    seed=42,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
)

trainer = Trainer(
    args=base_args,
    model_init=model_init,  # IMPORTANT: fresh model each trial
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

best_run = trainer.hyperparameter_search(
    backend="optuna",
    direction="maximize",
    hp_space=hp_space,
    compute_objective=compute_objective,
    n_trials=15,  # change to 30/50 if you can afford it
)

print("Best trial:", best_run)
best_cfg = best_run.hyperparameters
print("BEST CFG:", best_cfg)


# -----------------------
# G) FINAL TRAIN with best hyperparameters (then test ONCE)
# -----------------------
final_args = TrainingArguments(
    output_dir="dictabert_final",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    learning_rate=best_cfg["learning_rate"],
    weight_decay=best_cfg["weight_decay"],
    warmup_ratio=best_cfg["warmup_ratio"],
    per_device_train_batch_size=best_cfg["per_device_train_batch_size"],
    per_device_eval_batch_size=32,
    num_train_epochs=best_cfg["num_train_epochs"],
    report_to="none",
    seed=42,
)

final_trainer = Trainer(
    args=final_args,
    model_init=model_init,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

final_trainer.train()

print("\nTEST:")
test_metrics = final_trainer.evaluate(ds_tok["test"])
print(test_metrics)


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/11823 [00:00<?, ? examples/s]

Map:   0%|          | 0/1999 [00:00<?, ? examples/s]

...

Best trial: BestRun(run_id='5', objective=0.7493921613067256, hyperparameters={'learning_rate': 4.177040568759707e-05, 'weight_decay': 0.036956832426279354, 'per_device_train_batch_size': 16, 'warmup_ratio': 0.07339576952648837, 'num_train_epochs': 2}, run_summary=None)
BEST CFG: {'learning_rate': 4.177040568759707e-05, 'weight_decay': 0.036956832426279354, 'per_device_train_batch_size': 16, 'warmup_ratio': 0.07339576952648837, 'num_train_epochs': 2}


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dicta-il/dictabert and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.878900,0.669492,0.730865,0.730532
2,0.577800,0.735189,0.749375,0.749392



TEST:


{'eval_loss': 0.6567966938018799, 'eval_accuracy': 0.7760180995475113, 'eval_macro_f1': 0.7748273498415771, 'eval_runtime': 0.847, 'eval_samples_per_second': 1043.69, 'eval_steps_per_second': 33.058, 'epoch': 2.0}
